In [ ]:
%%sql -r dataframe_1
use database munchmarket;
use schema dbo;

CREATE OR REPLACE TABLE status_flags (
    datainflag INTEGER NOT NULL
);

INSERT INTO status_flags VALUES (0);


In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE PROCEDURE MUNCHMARKET.DBO.RUN_IF_FLAGGED()
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE v_flag INTEGER;
BEGIN
    SELECT datainflag INTO :v_flag
    FROM MUNCHMARKET.DBO.STATUS_FLAGS;

    IF (v_flag = 1) THEN

        -- Run the notebook
        EXECUTE NOTEBOOK MUNCHMARKET.DBO.BRONZE_COMPLETE();

        -- Run Silver notebook
        EXECUTE NOTEBOOK MUNCHMARKET.DBO.SILVER_COMPLETE();
        
        -- Run RESTAURANT DATA QUALITY notebook
        EXECUTE NOTEBOOK MUNCHMARKET.DBO.RESTAURANTDATAQUALITY();

        -- Run clean and tidy 
        EXECUTE NOTEBOOK MUNCHMARKET.DBO.CLEANANDTIDY();

        -- Run SALESFORCASTING:
        EXECUTE NOTEBOOK MUNCHMARKET.DBO.SALESFORCASTING();

        -- Reset flag to 0 when done
        UPDATE MUNCHMARKET.DBO.STATUS_FLAGS
        SET datainflag = 0;

        RETURN 'Flag was 1 → executed notebook + reset to 0';
    ELSE
        RETURN 'Flag was 0 → skipped';
    END IF;

END;
$$;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TASK t_check_flag_every_30m
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = '30 MINUTE'
AS
  CALL run_if_flagged();

In [ ]:
%%sql -r dataframe_4
CALL MUNCHMARKET.DBO.RUN_IF_FLAGGED();

In [ ]:
%%sql -r dataframe_6
UPDATE STATUS_FLAGS SET datainflag = 0;

In [ ]:
%%sql -r dataframe_5
-- In current context
SHOW NOTEBOOKS;

-- Scoped to a specific database or schema
SHOW NOTEBOOKS IN DATABASE munchmarket;
SHOW NOTEBOOKS IN SCHEMA munchmarket.dbo;